1. Why evaluate?

Evaluation means measuring how good your RAG is with numbers instead of by reading a few answers. Without numbers, you can't tell whether a change (new chunk size, new model) made things better or worse. You're only guessing.

RAG has two parts that fail differently, so we measure them separately:

Part	Question we ask	Measured with
Retrieval	Did we fetch the right chunks?	MRR, nDCG, hit rate
Generation	Is the answer correct and based on the chunks?	LLM-as-judge
2. Test data terms

Golden dataset (also called test set or eval set): a fixed list of test questions where you already know the right answer. It's like an exam with an answer key. It never changes between experiments, so results stay comparable.

Ground truth: the known-correct answer for a test question. Here it has two parts:

Expected source: which chunk should be retrieved (e.g. Service: CloudGuard)
Reference answer: what a correct answer says (e.g. "every 4 hours")

Test case: one row of the golden dataset, i.e. question + expected source + reference answer.

Question categories: groups of test questions by type (direct fact, needs two sections, no answer in the knowledge base, follow-up, and so on). They show where the RAG is weak, not just an overall score.

3. Retrieval terms

Rank: the position of a chunk in the retrieved list. #1 is the top result.

Relevant chunk: a chunk that actually contains the answer.

Top-k: how many chunks we retrieve (ours is 3).

Hit rate (also called Recall@k): did the right chunk appear anywhere in the top-k? It's either 1 (yes) or 0 (no), averaged over all questions. It's simple, but it ignores position: #1 and #3 count the same.

Reciprocal rank (RR): 1 divided by the rank of the first relevant chunk:

found at #1 → 1/1 = 1.0
found at #2 → 1/2 = 0.5
found at #3 → 1/3 = 0.33
not found → 0

MRR (Mean Reciprocal Rank): the average RR over all test questions. It rewards putting the right chunk near the top. It ranges from 0 (never found) to 1 (always #1).

Example with 3 questions:

Question	Right chunk at	RR
Backup	#1	1.0
"When did she join?" (no rewrite)	#3	0.33
Weather	not found	0

MRR = (1.0 + 0.33 + 0) ÷ 3 = 0.44

nDCG (normalised Discounted Cumulative Gain): like MRR, but it handles several relevant chunks and degrees of relevance. It's built up in steps:

Gain: how useful a chunk is, e.g. 1 = relevant, 0 = not relevant.
Discount: lower positions count less. Gain at rank r is divided by log₂(r + 1): rank 1 ÷ 1, rank 2 ÷ 1.58, rank 3 ÷ 2.
DCG: add up the discounted gains of your actual results.
IDCG (Ideal DCG): the DCG of the perfect ordering, with all relevant chunks at the top.
nDCG = DCG ÷ IDCG: a score from 0 to 1, where 1 means perfect ordering.

Example: "Who is Patel?" has two relevant chunks (Priya and Rahul). Suppose we retrieve [Priya, Sophie, Rahul]:

DCG = 1/1 + 0/1.58 + 1/2 = 1.5
Ideal order is [Priya, Rahul, other] → IDCG = 1/1 + 1/1.58 + 0 = 1.63
nDCG = 1.5 ÷ 1.63 = 0.92

MRR would give this a perfect 1.0, because it only looks at the first hit. nDCG notices that Rahul was ranked below an irrelevant chunk.

MRR vs nDCG: use MRR when there's one right chunk per question; use nDCG when several chunks matter.

Keyword coverage (the course uses this too): what fraction of expected key facts (e.g. "4 hours", "March 2019") appear in the retrieved chunks. A quick sanity check.

4. Generation (answer) terms

LLM-as-judge: using a second LLM to grade answers against the reference answer and context. It's faster and cheaper than human grading, but the judge can make mistakes too, so spot-check it.

Rubric: the grading criteria given to the judge. Typical ones:

Accuracy (correctness): does the answer match the reference answer?
Completeness: does it cover everything the question asked?
Relevance: does it stay on the question, without padding?
Faithfulness (groundedness): is every claim supported by the retrieved context? The "have your MFA device ready" answer from Day 1 would lose points here.

Hallucination: a claim that isn't supported by the context or by reality. Low faithfulness means hallucination.

Structured output: forcing the LLM to reply in a fixed format (fields with types) instead of free text, e.g. {"accuracy": 4, "faithfulness": 5, "feedback": "..."}. This lets you calculate averages in code, whereas free text can't be averaged.

Pydantic: a Python library for defining data shapes with types.

BaseModel: the class you inherit from to define a shape (e.g. class Grade(BaseModel):).
Field: adds a description or limits to a field (e.g. a score must be 1–5).
LangChain and OpenAI can use a Pydantic model to force the LLM's output into exactly that shape.

Score scale: e.g. 1–5 per rubric item, averaged across all test questions.

5. Experiment terms

Baseline: your current system's scores, the number to beat. (For us: ingest.py + answer.py as they are now.)

Experiment: change one thing, re-run the full eval, and compare with the baseline.

Ablation: removing or changing one component to see how much it mattered, e.g. "turn off query rewriting → how much does MRR drop?"

Things we'll vary:

Chunk size: how big each piece is (characters or tokens).
Chunk overlap: how much consecutive chunks share, so facts aren't cut in half at a boundary.
Chunking strategy: by headings (ours), by fixed size, or by meaning (semantic chunking, Day 5).
Embedding model: OpenAI text-embedding-3-small vs -large vs HuggingFace models.
Top-k: 3 vs 5 vs 10.

Trade-off: improving one number can hurt another. More top-k raises hit rate but adds noise and cost. Always report quality + latency + cost together.

## the golden dataset

In [1]:
from pydantic import BaseModel

from rag_knowledge_base_exploration.answer import retriever, answer_question


class TestCase(BaseModel):
    question: str
    category: str
    expected_sections: list[str]
    reference_answer: str


tests = [
    TestCase(
        question="How much does CloudGuard cost?",
        category="direct",
        expected_sections=["Service: CloudGuard"],
        reference_answer="£15 per user per month.",
    ),
    TestCase(
        question="What does Priya do?",
        category="direct",
        expected_sections=["Employee: Priya Patel"],
        reference_answer="Head of Service Desk; manages the 12-person service desk team, owns ticketing, reports monthly on response times.",
    ),
    TestCase(
        question="How often is my data backed up?",
        category="no_keyword",
        expected_sections=["Service: CloudGuard"],
        reference_answer="Every 4 hours.",
    ),
    TestCase(
        question="Who is in charge of technology?",
        category="no_keyword",
        expected_sections=["Employee: Daniel Hughes"],
        reference_answer="Daniel Hughes, the Chief Technology Officer.",
    ),
    TestCase(
        question="How do I connect to the VPN from home?",
        category="direct",
        expected_sections=["Policy: Remote Access (VPN)"],
        reference_answer="Connect through the company VPN, which requires MFA at every login.",
    ),
    TestCase(
        question="Who is Patel?",
        category="ambiguous",
        expected_sections=["Employee: Priya Patel", "Employee: Rahul Patel"],
        reference_answer="Two people: Priya Patel (Head of Service Desk) and Rahul Patel (Cloud Engineer).",
    ),
    TestCase(
        question="Can I get support at the weekend?",
        category="multi_section",
        expected_sections=["Frequently Asked Questions", "Support Hours and Response Times"],
        reference_answer="Only emergency support, and only for CloudGuard clients.",
    ),
    TestCase(
        question="What happens if I lose my laptop?",
        category="no_keyword",
        expected_sections=["Policy: Laptops and Devices"],
        reference_answer="Report it to the service desk within 1 hour.",
    ),
    TestCase(
        question="Which staff work in the Leeds office?",
        category="multi_section",
        expected_sections=["Employee: Rahul Patel", "Employee: Sophie Wright"],
        reference_answer="Rahul Patel and Sophie Wright.",
    ),
    TestCase(
        question="What is the company's parental leave policy?",
        category="no_answer",
        expected_sections=[],
        reference_answer="Not in the knowledge base; the assistant should say it doesn't know.",
    ),
]

print("Test cases:", len(tests))
for t in tests:
    print(f"  [{t.category}] {t.question}")

Test cases: 10
  [direct] How much does CloudGuard cost?
  [direct] What does Priya do?
  [no_keyword] How often is my data backed up?
  [no_keyword] Who is in charge of technology?
  [direct] How do I connect to the VPN from home?
  [ambiguous] Who is Patel?
  [multi_section] Can I get support at the weekend?
  [no_keyword] What happens if I lose my laptop?
  [multi_section] Which staff work in the Leeds office?
  [no_answer] What is the company's parental leave policy?


In [2]:
##run retrieval on every test question

retrieval_results = []

for t in tests:
    docs = retriever.invoke(t.question)

    retrieved_sections = []
    for doc in docs:
        retrieved_sections.append(doc.metadata["section"])

    retrieval_results.append(retrieved_sections)

    print(t.question)
    print("  expected: ", t.expected_sections)
    print("  retrieved:", retrieved_sections)
    print()

How much does CloudGuard cost?
  expected:  ['Service: CloudGuard']
  retrieved: ['Service: CloudGuard', 'Support Hours and Response Times', 'Service: SecureStart']

What does Priya do?
  expected:  ['Employee: Priya Patel']
  retrieved: ['Employee: Priya Patel', 'Employee: Sophie Wright', 'Employee: Rahul Patel']

How often is my data backed up?
  expected:  ['Service: CloudGuard']
  retrieved: ['Service: CloudGuard', 'Support Hours and Response Times', 'Frequently Asked Questions']

Who is in charge of technology?
  expected:  ['Employee: Daniel Hughes']
  retrieved: ['Employee: Daniel Hughes', 'Policy: Laptops and Devices', 'Employee: Rahul Patel']

How do I connect to the VPN from home?
  expected:  ['Policy: Remote Access (VPN)']
  retrieved: ['Policy: Remote Access (VPN)', 'Policy: Passwords', 'Frequently Asked Questions']

Who is Patel?
  expected:  ['Employee: Priya Patel', 'Employee: Rahul Patel']
  retrieved: ['Employee: Priya Patel', 'Employee: Rahul Patel', 'Employee: Emma 

In [3]:
###hit rate and MRR
def reciprocal_rank(expected, retrieved):
    for position in range(len(retrieved)):
        if retrieved[position] in expected:
            rank = position + 1
            return 1 / rank
    return 0


def hit(expected, retrieved):
    for section in retrieved:
        if section in expected:
            return 1
    return 0


rr_scores = []
hit_scores = []

for i in range(len(tests)):
    t = tests[i]
    retrieved = retrieval_results[i]

    if t.category == "no_answer":
        continue

    rr = reciprocal_rank(t.expected_sections, retrieved)
    h = hit(t.expected_sections, retrieved)
    rr_scores.append(rr)
    hit_scores.append(h)

    print(f"RR={rr:.2f}  hit={h}  [{t.category}] {t.question}")

mrr = sum(rr_scores) / len(rr_scores)
hit_rate = sum(hit_scores) / len(hit_scores)

print()
print(f"MRR:      {mrr:.3f}")
print(f"Hit rate: {hit_rate:.3f}")


RR=1.00  hit=1  [direct] How much does CloudGuard cost?
RR=1.00  hit=1  [direct] What does Priya do?
RR=1.00  hit=1  [no_keyword] How often is my data backed up?
RR=1.00  hit=1  [no_keyword] Who is in charge of technology?
RR=1.00  hit=1  [direct] How do I connect to the VPN from home?
RR=1.00  hit=1  [ambiguous] Who is Patel?
RR=1.00  hit=1  [multi_section] Can I get support at the weekend?
RR=1.00  hit=1  [no_keyword] What happens if I lose my laptop?
RR=1.00  hit=1  [multi_section] Which staff work in the Leeds office?

MRR:      1.000
Hit rate: 1.000


In [ ]:
##nDCG
import math


def dcg(relevances):
    total = 0
    for position in range(len(relevances)):
        rank = position + 1
        total = total + relevances[position] / math.log2(rank + 1)
    return total


def ndcg(expected, retrieved):
    relevances = []
    for section in retrieved:
        if section in expected:
            relevances.append(1)
        else:
            relevances.append(0)

    ideal = sorted(relevances, reverse=True)
    number_missing = min(len(expected), len(retrieved)) - sum(relevances)
    for _ in range(number_missing):
        ideal.pop()
        ideal.insert(0, 1)

    ideal_dcg = dcg(ideal)
    if ideal_dcg == 0:
        return 0
    return dcg(relevances) / ideal_dcg